# **Motion Tracking with Mean Shift and CAMSHIFT.**

#### **Object Tracking Algorithms:**
1. How to use the Mean Shift Algorithm in OpenCV
2. Use CAMSHIFT in OpenCV

> **Assets note.** The original asset pack for this course was never committed to this repo (no `images/`, `videos/`, or `haarcascades/`, in any past commit). This fork ships a real, working replacement set — sourced from OpenCV's own official sample data where a good match existed, generated procedurally where it didn't (see `scripts/generate_assets.py`) — under `assets/images/`, `assets/videos/`, and `assets/haarcascades/`, plus small, called-out fixes wherever a swap needed one. Nothing else was removed. See the README for the full source list. No download step needed — just run the cells top to bottom.

In [1]:
import cv2
import numpy as np
from matplotlib import pyplot as plt

# Define our imshow function 
def imshow(title = "Image", image = None, size = 10):
    w, h = image.shape[0], image.shape[1]
    aspect_ratio = w/h
    plt.figure(figsize=(size * aspect_ratio,size))
    plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    plt.title(title)
    plt.show()

## **Meanshift Object Tracking** 

The intuition behind the meanshift is simple. Consider you have a set of points. (It can be a pixel distribution like histogram backprojection). You are given a small window ( may be a circle) and you have to move that window to the area of maximum pixel density (or maximum number of points). 

Mean shift is a hill climbing algorithm which involves shifting this kernel iteratively to a higher density region until convergence. Every shift is defined by a mean shift vector. The mean shift vector always points toward the direction of the maximum increase in the density. 


![](https://upload.wikimedia.org/wikipedia/commons/b/bd/Meanshiftred.gif)

Read Paper Here - https://ieeexplore.ieee.org/document/732882

Animation Source - https://fr.wikipedia.org/wiki/Camshift

**Concretely, one meanshift step is:** treat the pixel values inside the current search window as a weighted cloud of points (weighted by `dst`, the histogram-backprojection score — literally "how much does this pixel's hue look like our tracked object's hue", computed a few cells down). Compute that cloud's **weighted centroid** — its center of mass, same idea as notebook 12's contour centroid, just weighted by backprojection score instead of "is this pixel white". Move the window so it's centered on that centroid. Repeat. Since the object being tracked is (by construction) the brightest cluster of backprojection score nearby, each step drags the window a little closer to sitting exactly on top of it — like a marble rolling downhill into the nearest dip, except here it's climbing *uphill* into the nearest density peak. `term_crit` caps this at 10 iterations or a 1-pixel move, whichever comes first, per frame.

**CAMSHIFT** (below) is meanshift plus one addition: after converging, it also looks at the *spread* (second moment) of the matched region to resize and rotate the tracking window to fit — which is why its output is a rotated box (`cv2.boxPoints`) instead of meanshift's fixed-size, axis-aligned rectangle.

In [2]:
cap = cv2.VideoCapture('./assets/videos/data_slow.flv')  # real traffic clip — see the README for source/license

# take first frame of the video
ret,frame = cap.read()

# Get the height and width of the frame (required to be an interger)
width = int(cap.get(3)) 
height = int(cap.get(4))

# Define the codec and create VideoWriter object. The output is stored in '*.avi' file.
out = cv2.VideoWriter('car_tracking_mean_shift.avi', cv2.VideoWriter_fourcc('M','J','P','G'), 30, (width, height))

# setup initial location of window
r,h,c,w = 200,50,300,100  # the exact ROI OpenCV's own meanshift/CAMSHIFT tutorials use for this clip
track_window = (c,r,w,h)

# set up the ROI for tracking
roi = frame[r:r+h, c:c+w]
hsv_roi = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)  # from roi, not the whole frame
mask = cv2.inRange(hsv_roi, np.array((0., 60.,32.)), np.array((180.,255.,255.)))
roi_hist = cv2.calcHist([hsv_roi],[0],mask,[180],[0,180])
cv2.normalize(roi_hist,roi_hist,0,255,cv2.NORM_MINMAX)

# Setup the termination criteria, either 10 iteration or move by atleast 1 pt
term_crit = ( cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 10, 1 )

while(1):
    ret, frame = cap.read()

    if ret == True:
        hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
        dst = cv2.calcBackProject([hsv],[0],roi_hist,[0,180],1)

        # apply meanshift to get the new location
        ret, track_window = cv2.meanShift(dst, track_window, term_crit)

        # Draw it on image
        x,y,w,h = track_window
        img2 = cv2.rectangle(frame, (x,y), (x+w,y+h), (255,255,255),2)
        out.write(img2)
        #imshow('Tracking', img2)

    else:
        break

cap.release()
out.release()

In [3]:
!ffmpeg -i ./car_tracking_mean_shift.avi car_tracking_mean_shift.mp4 -y

ffmpeg version 8.1.1 Copyright (c) 2000-2026 the FFmpeg developers
  built with Apple clang version 21.0.0 (clang-2100.0.123.102)
  configuration: --prefix=/opt/homebrew/Cellar/ffmpeg/8.1.1 --enable-shared --enable-pthreads --enable-version3 --cc=clang --host-cflags= --host-ldflags= --enable-ffplay --enable-gpl --enable-libsvtav1 --enable-libopus --enable-libx264 --enable-libmp3lame --enable-libdav1d --enable-libvmaf --enable-libvpx --enable-libx265 --enable-openssl --enable-videotoolbox --enable-audiotoolbox --enable-neon
  libavutil      60. 26.101 / 60. 26.101
  libavcodec     62. 28.101 / 62. 28.101
  libavformat    62. 12.101 / 62. 12.101
  libavdevice    62.  3.101 / 62.  3.101
  libavfilter    11. 14.101 / 11. 14.101
  libswscale      9.  5.101 /  9.  5.101
  libswresample   6.  3.101 /  6.  3.101
Input #0, avi, from './car_tracking_mean_shift.avi':
  Metadata:
    software        : Lavf60.3.100
  Duration: 00:00:30.43, start: 0.000000, bitrate: 5013 kb/s
  Stream #0:0: Video: m

[out#0/mp4 @ 0x78cc34180] video:1775KiB audio:0KiB subtitle:0KiB other streams:0KiB global headers:0KiB muxing overhead: 0.650375%
frame=  913 fps=0.0 q=-1.0 Lsize=    1787KiB time=00:00:30.36 bitrate= 482.1kbits/s speed=35.8x elapsed=0:00:00.84    
[libx264 @ 0x78d02ca80] frame I:4     Avg QP:19.25  size: 16344
[libx264 @ 0x78d02ca80] frame P:230   Avg QP:21.51  size:  4728
[libx264 @ 0x78d02ca80] frame B:679   Avg QP:26.26  size:   979
[libx264 @ 0x78d02ca80] consecutive B-frames:  0.5%  0.7%  0.7% 98.1%
[libx264 @ 0x78d02ca80] mb I  I16..4:  8.6% 84.2%  7.2%
[libx264 @ 0x78d02ca80] mb P  I16..4:  3.2% 16.3%  0.7%  P16..4: 26.6% 17.0% 12.0%  0.0%  0.0%    skip:24.2%
[libx264 @ 0x78d02ca80] mb B  I16..4:  0.6%  1.4%  0.0%  B16..8: 38.1%  5.6%  0.7%  direct: 1.4%  skip:52.2%  L0:47.7% L1:48.0% BI: 4.3%
[libx264 @ 0x78d02ca80] 8x8 transform intra:78.4% inter:76.1%
[libx264 @ 0x78d02ca80] coded y,uvDC,uvAC intra: 43.9% 65.7% 6.5% inter: 8.5% 11.4% 0.1%
[libx264 @ 0x78d02ca80] i16 v,h,dc,

In [4]:
from IPython.display import HTML
from base64 import b64encode

mp4 = open('./car_tracking_mean_shift.mp4','rb').read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()

In [5]:
HTML("""
<video controls>
      <source src="%s" type="video/mp4">
</video>
""" % data_url)

## **Camshift in OpenCV** 
It is almost same as meanshift, but it returns a rotated rectangle (that is our result) and box parameters (used to be passed as search window in next iteration). 

![](https://upload.wikimedia.org/wikipedia/commons/8/86/CamshiftStillImage.gif)

Read Paper Here - https://ieeexplore.ieee.org/document/732882

Animation Source - https://fr.wikipedia.org/wiki/Camshift

In [6]:
cap = cv2.VideoCapture('./assets/videos/data_slow.flv')  # real traffic clip — see the README for source/license

# take first frame of the video
ret,frame = cap.read()

# Get the height and width of the frame (required to be an interger)
width = int(cap.get(3)) 
height = int(cap.get(4))

# Define the codec and create VideoWriter object. The output is stored in '*.avi' file.
out = cv2.VideoWriter('car_tracking_cam_shift.avi', cv2.VideoWriter_fourcc('M','J','P','G'), 30, (width, height))

# setup initial location of window
r,h,c,w = 200,50,300,100  # the exact ROI OpenCV's own meanshift/CAMSHIFT tutorials use for this clip
track_window = (c,r,w,h)

# set up the ROI for tracking
roi = frame[r:r+h, c:c+w]
hsv_roi = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)  # from roi, not the whole frame
mask = cv2.inRange(hsv_roi, np.array((0., 60.,32.)), np.array((180.,255.,255.)))
roi_hist = cv2.calcHist([hsv_roi],[0],mask,[180],[0,180])
cv2.normalize(roi_hist,roi_hist,0,255,cv2.NORM_MINMAX)

# Setup the termination criteria, either 10 iteration or move by atleast 1 pt
term_crit = ( cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 10, 1 )

while(1):
    ret ,frame = cap.read()

    if ret == True:
        hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
        dst = cv2.calcBackProject([hsv],[0],roi_hist,[0,180],1)

        # apply meanshift to get the new location
        ret, track_window = cv2.CamShift(dst, track_window, term_crit)

        # Draw it on image
        pts = cv2.boxPoints(ret)
        pts = np.intp(pts)  # np.int0 was removed in NumPy 2.0
        img2 = cv2.polylines(frame,[pts],True, 255,2)
        out.write(img2)
        #imshow('img2',img2)

    else:
        break

cap.release()
out.release()

In [7]:
!ffmpeg -i ./car_tracking_cam_shift.avi car_tracking_cam_shift.mp4 -y

ffmpeg version 8.1.1 Copyright (c) 2000-2026 the FFmpeg developers
  built with Apple clang version 21.0.0 (clang-2100.0.123.102)
  configuration: --prefix=/opt/homebrew/Cellar/ffmpeg/8.1.1 --enable-shared --enable-pthreads --enable-version3 --cc=clang --host-cflags= --host-ldflags= --enable-ffplay --enable-gpl --enable-libsvtav1 --enable-libopus --enable-libx264 --enable-libmp3lame --enable-libdav1d --enable-libvmaf --enable-libvpx --enable-libx265 --enable-openssl --enable-videotoolbox --enable-audiotoolbox --enable-neon
  libavutil      60. 26.101 / 60. 26.101
  libavcodec     62. 28.101 / 62. 28.101
  libavformat    62. 12.101 / 62. 12.101
  libavdevice    62.  3.101 / 62.  3.101
  libavfilter    11. 14.101 / 11. 14.101
  libswscale      9.  5.101 /  9.  5.101
  libswresample   6.  3.101 /  6.  3.101
Input #0, avi, from './car_tracking_cam_shift.avi':
  Metadata:
    software        : Lavf60.3.100
  Duration: 00:00:30.43, start: 0.000000, bitrate: 5322 kb/s
  Stream #0:0: Video: mj

[out#0/mp4 @ 0xc6300c840] video:2095KiB audio:0KiB subtitle:0KiB other streams:0KiB global headers:0KiB muxing overhead: 0.551786%
frame=  913 fps=0.0 q=-1.0 Lsize=    2107KiB time=00:00:30.36 bitrate= 568.4kbits/s speed=34.4x elapsed=0:00:00.88    
[libx264 @ 0xc62c30a80] frame I:4     Avg QP:19.36  size: 16968
[libx264 @ 0xc62c30a80] frame P:230   Avg QP:21.77  size:  5225
[libx264 @ 0xc62c30a80] frame B:679   Avg QP:26.69  size:  1289
[libx264 @ 0xc62c30a80] consecutive B-frames:  0.8%  0.2%  0.0% 99.0%
[libx264 @ 0xc62c30a80] mb I  I16..4:  8.5% 83.6%  7.9%
[libx264 @ 0xc62c30a80] mb P  I16..4:  3.3% 16.3%  1.0%  P16..4: 26.8% 17.4% 12.3%  0.0%  0.0%    skip:22.9%
[libx264 @ 0xc62c30a80] mb B  I16..4:  0.7%  1.5%  0.1%  B16..8: 38.4%  6.4%  1.1%  direct: 1.5%  skip:50.4%  L0:47.9% L1:47.8% BI: 4.2%
[libx264 @ 0xc62c30a80] 8x8 transform intra:76.2% inter:74.6%
[libx264 @ 0xc62c30a80] coded y,uvDC,uvAC intra: 44.5% 67.3% 10.3% inter: 9.1% 13.4% 2.0%
[libx264 @ 0xc62c30a80] i16 v,h,dc

In [8]:
from IPython.display import HTML
from base64 import b64encode

mp4 = open('car_tracking_cam_shift.mp4','rb').read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()

In [9]:
HTML("""
<video controls>
      <source src="%s" type="video/mp4">
</video>
""" % data_url)

### If you want to push this further

- CAMSHIFT adapts window *size* as the tracked region's histogram back-projection changes; meanshift's window is fixed. Print `track_window` each frame for both and compare how much the box size moves for CAMSHIFT vs. staying constant for meanshift.
- Both algorithms track by hue histogram alone — repaint the synthetic car a color close to the road (`scripts/generate_assets.py`, `CAR_COLOR`) and watch tracking degrade as contrast with the background drops.